### Analysis of flow

In [ ]:
#read in:
import pandas as pd 
import numpy as np
from scipy import stats 

raw_df = pd.read_excel("RITA_Flow_Data.xlsx", sheet_name='Sheet5', header=0)

#visualize

print(raw_df.head())

In [ ]:
#assuming dead = LR + UR 

raw_df['Percent Dead'] = raw_df['LR'] + raw_df['UR']

#remove a replicate, x here. If you're reading this and wondering why we dropped these reps, it is due to issues with the killing in these reps, suggesting that generally the expectation
# that more cells will die with RITA than DMSO is confounded in a way that is clearly an outlier of the biology. Therefore we drop them due to this inconsistency. 

rep_to_remove = [3, 4, 5]
for rep in rep_to_remove:
    raw_df = raw_df[raw_df['rep']!=rep]
    
print(f"REMOVED REPLICATE {rep_to_remove}")
#now we need to drop all treatment abt and sample T10

raw_df = raw_df[(raw_df['sample'] != 'T10') | (raw_df['treatment'] != 'ABT')]

#now that its cleaned, we want to get two dictionaries: (1) Control DMSO, (2) Control RITA
#where each takes rep as input and as output a dead amount

map_controlDMSO = {}
map_controlRITA ={}
map_control_difference = {}

#all unique reps:

reps = raw_df['rep'].unique()




#for each rep, lets populate the sample N174 with either treatment DMSO or RITA

for rep in reps:
    DMSO_dead_control = raw_df[(raw_df['rep']==rep) & (raw_df['sample'].str.contains('N174')) & (raw_df['treatment'].str.contains('DMSO'))]
    RITA_dead_control = raw_df[(raw_df['rep']==rep) & (raw_df['sample'].str.contains('N174')) &(raw_df['treatment'].str.contains('RITA'))]
    
    #these are currently dataframes
    #we have baseline for dmso and rita
    #if multiple per rep, then we need to deal with that by taking averages
    DMSO_dead_control_value = DMSO_dead_control['Percent Dead'].mean()
    RITA_dead_control_value = RITA_dead_control['Percent Dead'].mean()
    
    #now put into maps...
    
    map_controlDMSO[rep] = DMSO_dead_control_value
    map_controlRITA[rep] = RITA_dead_control_value
    map_control_difference[rep] = map_controlRITA[rep] - map_controlDMSO[rep]
        



In [ ]:
#now that we have the maps, we can filter out the controls.

control_filtered_df = raw_df[~raw_df['sample'].str.contains('N174')]
control_filtered_df = control_filtered_df[~control_filtered_df['sample'].str.contains('T12')]



In [ ]:
#set up new dataframe to hold data:

""" 
    Replicate | Peptide | DMSO | RITA | Difference | Control Difference | Recovery

"""

difference_dataframe = pd.DataFrame(columns=['Replicate', "Peptide", "Dead in DMSO", "Dead in RITA", "Increase in Death with RITA for this Peptide", "Increase in Death with RITA for replicate N174", "Recovery"])



In [ ]:
#now we need to get the normalized for each replicate, for each peptide tested.



for Replicate in reps:
    for Peptide in control_filtered_df['sample'].unique():
        
        DMSO_value = control_filtered_df[(control_filtered_df['rep']==Replicate) & (control_filtered_df['sample'] == Peptide) & (control_filtered_df['treatment'] == 'DMSO')]['Percent Dead'].mean()
        RITA_value = control_filtered_df[(control_filtered_df['rep']==Replicate) & (control_filtered_df['sample'] == Peptide) & (control_filtered_df['treatment'] == 'RITA')]['Percent Dead'].mean()

        Difference =  RITA_value - DMSO_value
        
        Control_Difference = map_control_difference[Replicate]
        
        Recovery =  Control_Difference - Difference
        
        difference_dataframe.loc[len(difference_dataframe)] = [Replicate, Peptide, DMSO_value, RITA_value, Difference, Control_Difference, Recovery]
        

#now we add the N174 separately...

        

In [ ]:
print(difference_dataframe.head())

In [ ]:
#okay now we want to go ahead and do the stats and visualization...

#the p value column will describe a one-sided t test in which there is a null hypothesis that the difference in control between DMSO and RITA induced cell death is greater than that of the peptide...

pvalue_map = {}

#Rep --> pvalue of t-test

#lets loop thru each peptide 
for Peptide in control_filtered_df['sample'].unique():
    #get the distribution of differences for the control from our dictionary from way earlier (see above)
    control_difference_distribution = list(map_control_difference.values())
    #now get the distribution of differences for the given peptide from the dataframe we have been working with...
    peptide_difference_distribution = list(difference_dataframe[difference_dataframe['Peptide']==Peptide]['Increase in Death with RITA for this Peptide'])
    # 1. Create a boolean mask of where NaNs are NOT present
    nan_indices = [i for i, x in enumerate(peptide_difference_distribution) if isinstance(x, float) and np.isnan(x)]

    # 2. Filter the target list (Filter OUT the positions)
    control_difference_distribution = [val for i, val in enumerate(control_difference_distribution) if i not in nan_indices]
    peptide_difference_distribution = [val for i, val in enumerate(peptide_difference_distribution) if i not in nan_indices]

    pvalue = stats.ttest_rel(peptide_difference_distribution, control_difference_distribution, alternative='less')[1]
    

    
    pvalue_map[Peptide] = pvalue
    



In [ ]:
#add n174 for visualization later...

for Replicate in list(map_control_difference.keys()):
    difference_dataframe.loc[len(difference_dataframe)] =[Replicate, 'N174', map_controlDMSO[Replicate], map_controlRITA[Replicate], map_control_difference[Replicate], np.nan, np.nan]
    

In [ ]:
#now lets visualize




import seaborn as sns 
import matplotlib.pyplot as plt 

#for coloring...

sns.color_palette("Set2")

from matplotlib import font_manager

from matplotlib import rcParams
rcParams['font.sans-serif'] = "Arial"
rcParams['font.family'] = "Arial"


fonts = sorted(font_manager.get_font_names())

print("Arial" in fonts)
print("Verdana" in fonts)
print("Times New Roman" in fonts)

# optional
print([f for f in fonts if "Arial" in f])

def assign_color(sample):
    if 'N174' in sample:
        return "N174"
    else:
        return "Peptide"

difference_dataframe['color_label'] = difference_dataframe['Peptide'].apply(lambda x: assign_color(x))

fig, ax = plt.subplots(figsize=(5,4))



custom_pallette = {"N174": 'gray', "Peptide":"#EC718B"}
sns.boxplot(data=difference_dataframe, y='Increase in Death with RITA for this Peptide', x='Peptide', fliersize=0, fill=False,color='black', linewidth=0.8,ax=ax, meanline=True, showmeans=True, meanprops={'color': 'k', 'ls': '-', 'lw': 2}, 
                        medianprops={'visible': False},
            whiskerprops={'visible': False},
            zorder=10,
            showfliers=False,
            showbox=False,
            showcaps=False,
            width=0.3
            )
sns.stripplot(data=difference_dataframe, y='Increase in Death with RITA for this Peptide', x='Peptide', jitter=False,size=4.5,edgecolor='black',ax=ax, alpha=0.9, hue='color_label', palette=custom_pallette,  legend=False)



#sns.barplot(data=difference_dataframe, y='Increase in Death with RITA for this Peptide', x='Peptide',ax=ax, alpha=0.9, hue='color_label', palette=custom_pallette, fill=False, errorbar=None, linewidth=2.0)

plt.ylabel('Increase in Cell Death by RITA')
plt.xlabel('Treatment')
plt.ylim(0,65)
plt.ylim(0, 70)

sns.despine(offset=10, trim=True)


#text label assignments. 
def get_tick_pos_by_name(ax, target_name):
    # Zip locations and labels together
    for pos, label in zip(ax.get_xticks(), ax.get_xticklabels()):
        if label.get_text() == target_name:
            return pos
    return None

rename_map = {
    '8': '7_1',
    '9': '7_2',
    '10': '7_3' # Third component example
}

#sns.move_legend(ax, 'upper right')
#plt.legend(title="")
# 2. Then directly modify the frame properties
leg = ax.get_legend()
if leg:
    leg.get_frame().set_edgecolor('white')
    leg.get_frame().set_facecolor('white') # Optional: Match background
    leg.get_frame().set_alpha(1) # Ensure solid
    


#the idea is that we need to loop thru each peptide, get its upper quartile 
for peptide in difference_dataframe['Peptide'].unique():
    y_position = 73
    try:
        x_position = get_tick_pos_by_name(ax, peptide) - 0.44
        offset = 0
        y_position += offset 
        pvalue = pvalue_map[peptide] 
        print(f"xposition: {x_position} yposition: {y_position}")

        fontsize = 8
        if pvalue < 0.05:
            ax.text(x=x_position, y=y_position, s = str(round(pvalue, 3)),fontsize=fontsize, color="#EC718B")
            #ax.text(x=x_position, y=y_position, s = "*",fontsize=fontsize, color='black')
        else:
            ax.text(x=x_position, y=y_position,fontsize=fontsize, s = str(round(pvalue, 3)),color="#A5A8A8")

    except: 
        None
label_map = {"N174": "N174", "T01": "T01","T02": "T02","T03": "T03","T04": "T04","T05": "T05","T06": "T06","T07": "T07","T08": "T08","T09": "T09","T10": "T10","A2": "T11","B6": "T12"}

current_labels = [tick.get_text() for tick in ax.get_xticklabels()]
new_labels = [label_map.get(label, label) for label in current_labels]
ax.set_xticklabels(new_labels)

#save 

#ax.text(x = 3, y = 70, s= f"p value | reps removed: {rep_to_remove}", fontsize = 14, color='black')
ax.text(x=-1.8, y=73.2, s="p-value", fontsize=8)
plt.tight_layout()
plt.savefig("RITA_Flow.pdf", dpi=300)


In [ ]:
fig, ax = plt.subplots()

# Manual approach: separate DMSO and RITA, then merge
dmso_df = new_df_raw[new_df_raw['treatment'] == 'DMSO'].copy()
rita_df = new_df_raw[new_df_raw['treatment'] == 'RITA'].copy()

# Average duplicates if any
dmso_df = dmso_df.groupby(['rep', 'sample'])['Percent Dead'].mean().reset_index()
dmso_df.columns = ['rep', 'sample', 'DMSO']

rita_df = rita_df.groupby(['rep', 'sample'])['Percent Dead'].mean().reset_index()
rita_df.columns = ['rep', 'sample', 'RITA']

# Merge on rep and sample
plot_ready_df = dmso_df.merge(rita_df, on=['rep', 'sample'], how='inner')

print("\nAfter merge:")
print("Unique reps:", sorted(plot_ready_df['rep'].unique()))
print("Total rows:", len(plot_ready_df))
print("\nSample of merged data:")
print(plot_ready_df.head(10))

# Convert rep to string for distinct colors
plot_ready_df['rep'] = plot_ready_df['rep'].astype(str)

sns.scatterplot(data=plot_ready_df, x='DMSO', y='RITA', hue='rep', ax=ax, s=15,palette='viridis')
plt.ylabel('% Dead with RITA')
plt.xlabel('% Dead with DMSO')
sns.move_legend(ax, 'upper left', bbox_to_anchor=(1,1))
plt.title("Percent dead with RITA by Percent Dead with DMSO, colored by replicate")
plt.show()

In [ ]:
#grubbs..

from outliers import smirnov_grubbs as grubbs
import pandas as pd



for peptide in difference_dataframe['Peptide'].unique():
    distribution_pep = difference_dataframe[difference_dataframe['Peptide'] == peptide]["Increase in Death with RITA for this Peptide"]
    distribution_pep = distribution_pep.dropna()
    print(f"Looking at peptide {peptide}")
    print(f"Distribution before: {list(distribution_pep)} ")
    print(f"distribution after: {list(grubbs.test(distribution_pep, alpha=0.05))}")
